# โฟลว์งาน Human-in-the-Loop กับ Microsoft Agent Framework

## 🎯 วัตถุประสงค์การเรียนรู้

ในสมุดบันทึกนี้ คุณจะได้เรียนรู้วิธีการใช้โฟลว์งาน **human-in-the-loop** โดยใช้ Microsoft Agent Framework กับ `RequestInfoExecutor` รูปแบบที่ทรงพลังนี้ช่วยให้คุณหยุดโฟลว์งาน AI เพื่อรับข้อมูลจากมนุษย์ ทำให้องค์กรของคุณมีการโต้ตอบและมนุษย์เป็นผู้ควบคุมการตัดสินใจที่สำคัญ

## 🔄 Human-in-the-Loop คืออะไร?

**Human-in-the-loop (HITL)** คือรูปแบบการออกแบบที่เอเย่นต์ AI จะหยุดการทำงานเพื่อขอข้อมูลจากมนุษย์ก่อนดำเนินการต่อ ซึ่งมีความสำคัญต่อ:

- ✅ **การตัดสินใจที่สำคัญ** - ขอการอนุมัติจากมนุษย์ก่อนดำเนินการสำคัญ
- ✅ **สถานการณ์ที่ไม่ชัดเจน** - ให้มนุษย์ช่วยชี้แจงเมื่อ AI ไม่แน่ใจ
- ✅ **ความชอบของผู้ใช้** - ถามผู้ใช้ให้เลือกตัวเลือกหลายตัว
- ✅ **การปฏิบัติตามและความปลอดภัย** - ให้มนุษย์ตรวจสอบสำหรับการดำเนินการที่ต้องควบคุม
- ✅ **ประสบการณ์แบบโต้ตอบ** - สร้างเอเย่นต์สนทนาที่ตอบสนองต่อข้อมูลผู้ใช้

## 🏗️ วิธีการทำงานใน Microsoft Agent Framework

เฟรมเวิร์กนี้มีส่วนประกอบหลักสามอย่างสำหรับ HITL:

1. **`RequestInfoExecutor`** - ตัวดำเนินการพิเศษที่หยุดโฟลว์งานและส่ง `RequestInfoEvent`
2. **`RequestInfoMessage`** - คลาสฐานสำหรับข้อมูลคำขอประเภทต่าง ๆ ที่ส่งไปยังมนุษย์
3. **`RequestResponse`** - เชื่อมโยงคำตอบของมนุษย์กับคำขอเดิมโดยใช้ `request_id`

**รูปแบบโฟลว์งาน:**
```
Agent detects need for input
    ↓
Sends message to RequestInfoExecutor
    ↓
Workflow pauses & emits RequestInfoEvent
    ↓
Application collects human input (console, UI, etc.)
    ↓
Application sends RequestResponse via send_responses_streaming()
    ↓
Workflow resumes with human input
```

## 🏨 ตัวอย่างของเรา: การจองโรงแรมพร้อมการยืนยันจากผู้ใช้

เราจะต่อยอดโฟลว์งานตามเงื่อนไขโดยเพิ่มการยืนยันจากมนุษย์ **ก่อน** เสนอจุดหมายปลายทางทางเลือก:

1. ผู้ใช้ร้องขอจุดหมาย (เช่น "ปารีส")
2. `availability_agent` ตรวจสอบว่ามีห้องว่างหรือไม่
3. **ถ้าไม่มีห้อง** → `confirmation_agent` ถามว่า "คุณต้องการดูตัวเลือกอื่นไหม?"
4. โฟลว์งาน **หยุดชั่วคราว** โดยใช้ `RequestInfoExecutor`
5. **มนุษย์ตอบกลับ** ด้วย "ใช่" หรือ "ไม่" ผ่านทางคอนโซล
6. `decision_manager` นำทางตามคำตอบ:
   - **ใช่** → แสดงจุดหมายทางเลือก
   - **ไม่** → ยกเลิกร้องขอการจอง
7. แสดงผลลัพธ์สุดท้าย

นี่เป็นตัวอย่างการให้ผู้ใช้ควบคุมคำแนะนำของเอเย่นต์!

---

มาเริ่มกันเลย! 🚀


## ขั้นตอนที่ 1: นำเข้าห้องสมุดที่จำเป็น

เรานำเข้าคอมโพเนนต์มาตรฐานของ Agent Framework รวมถึง **คลาสเฉพาะสำหรับ human-in-the-loop**:
- `RequestInfoExecutor` - ตัวปฏิบัติการที่หยุดชั่วคราวกระบวนการทำงานเพื่อรอข้อมูลจากมนุษย์
- `RequestInfoEvent` - เหตุการณ์ที่ถูกส่งออกเมื่อมีการร้องขอข้อมูลจากมนุษย์
- `RequestInfoMessage` - คลาสฐานสำหรับ payload คำขอที่มีชนิดเจาะจง
- `RequestResponse` - เชื่อมโยงคำตอบจากมนุษย์กับคำขอ
- `WorkflowOutputEvent` - เหตุการณ์สำหรับตรวจจับผลลัพธ์ของกระบวนการทำงาน


In [ ]:
import asyncio
import json
import os
from dataclasses import dataclass
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    Executor,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowRunState,          # Enum of workflow run states
    executor,
    handler,
    response_handler,          # NEW: decorator for the human-feedback response handler
    tool,
)

from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")
print("🔄 Human-in-the-loop uses ctx.request_info() + @response_handler")

✅ All imports successful!
🔄 Human-in-the-loop components loaded: RequestInfoExecutor, RequestInfoEvent, RequestResponse


## ขั้นตอนที่ 2: กำหนดโมเดล Pydantic สำหรับผลลัพธ์ที่มีโครงสร้าง

โมเดลเหล่านี้กำหนด **สกีมา** ที่เอเย่นต์จะส่งกลับ เราเก็บโมเดลทั้งหมดจากเวิร์กโฟลว์ตามเงื่อนไขและเพิ่ม:

**ใหม่สำหรับ Human-in-the-Loop:**
- `HumanFeedbackRequest` - ชนิดย่อยของ `RequestInfoMessage` ที่กำหนดข้อมูลคำขอที่ส่งไปยังมนุษย์
  - ประกอบด้วย `prompt` (คำถามที่จะถาม) และ `destination` (บริบทเกี่ยวกับเมืองที่ไม่สามารถเข้าถึงได้)


In [ ]:
# Existing models from conditional workflow
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""
    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""
    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""
    destination: str
    action: str
    message: str


# NEW: Pydantic model for agent's response format
class ConfirmationQuestion(BaseModel):
    """
    Pydantic model used by confirmation_agent's response_format.
    This is what the agent will output as JSON.
    """
    question: str  # The question to ask the user
    destination: str  # The unavailable destination for context


# NEW: Plain dataclass payload for ctx.request_info()
@dataclass
class HumanFeedbackRequest:
    """
    Request sent to RequestInfoExecutor asking if user wants alternatives.
    
    MUST be a dataclass subclassing RequestInfoMessage for type compatibility.
    This is what gets sent to the RequestInfoExecutor.
    """
    prompt: str = ""  # The question to ask the user
    destination: str = ""  # The unavailable destination for context


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")
print("   - ConfirmationQuestion (agent response format) 🆕")
print("   - HumanFeedbackRequest (RequestInfoMessage for HITL) 🆕")

✅ Pydantic models defined:
   - BookingCheckResult (availability check)
   - AlternativeResult (alternative suggestion)
   - BookingConfirmation (booking confirmation)
   - ConfirmationQuestion (agent response format) 🆕
   - HumanFeedbackRequest (RequestInfoMessage for HITL) 🆕


## ขั้นตอนที่ 3: สร้างเครื่องมือจองโรงแรม

เครื่องมือเดียวกันกับในกระบวนการทำงานแบบมีเงื่อนไข - ตรวจสอบว่าห้องพักว่างในจุดหมายปลายทางหรือไม่


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.
    
    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

✅ hotel_booking tool created with @ai_function decorator


## ขั้นตอนที่ 4: กำหนดฟังก์ชันเงื่อนไขสำหรับการกำหนดเส้นทาง

เราต้องการ **ฟังก์ชันเงื่อนไขสี่ฟังก์ชัน** สำหรับเวิร์กโฟลว์แบบ human-in-the-loop ของเรา:

**จากเวิร์กโฟลว์แบบมีเงื่อนไข:**
1. `has_availability_condition` - กำหนดเส้นทางเมื่อโรงแรมมีห้องว่าง
2. `no_availability_condition` - กำหนดเส้นทางเมื่อโรงแรมไม่มีห้องว่าง

**ใหม่สำหรับ human-in-the-loop:**
3. `user_wants_alternatives_condition` - กำหนดเส้นทางเมื่อผู้ใช้ตอบ "ใช่" สำหรับทางเลือกอื่น
4. `user_declines_alternatives_condition` - กำหนดเส้นทางเมื่อผู้ใช้ตอบ "ไม่" สำหรับทางเลือกอื่น


In [24]:
# Existing condition functions from conditional workflow
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )
        return result.has_availability
    except Exception as e:
        display(HTML(f"""<div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'><strong>⚠️  Error:</strong> {str(e)}</div>"""))
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )
        return not result.has_availability
    except Exception as e:
        return False


# NEW: Condition functions for human-in-the-loop routing
def user_wants_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user WANTS to see alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            wants_alternatives = "wants to see alternative" in msg_text or "want to see alternative" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
                    <strong>🔍 User Decision:</strong> User wants alternatives = <strong>{wants_alternatives}</strong>
                </div>
            """)
            )
            
            return wants_alternatives
    
    return False
def user_declines_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user DECLINES alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            declined = "declined" in msg_text or "has declined" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fce4ec; border-left: 4px solid #c2185b; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 User Decision:</strong> User declined alternatives = <strong>{declined}</strong>
                </div>
            """)
            )
            
            return declined
    
    return False
print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")
print("   - user_wants_alternatives_condition (routes when user says yes) 🆕")
print("   - user_declines_alternatives_condition (routes when user says no) 🆕")

✅ Condition functions defined:
   - has_availability_condition (routes when rooms exist)
   - no_availability_condition (routes when no rooms)
   - user_wants_alternatives_condition (routes when user says yes) 🆕
   - user_declines_alternatives_condition (routes when user says no) 🆕


## ขั้นตอนที่ 5: สร้าง Decision Manager Executor

นี่คือ **แกนหลักของรูปแบบ human-in-the-loop**! `DecisionManager` เป็น `Executor` ที่กำหนดเองซึ่ง:

1. **รับข้อเสนอแนะจากมนุษย์** ผ่านอ็อบเจ็กต์ `RequestResponse`
2. **ประมวลผลการตัดสินใจของผู้ใช้** (ใช่/ไม่ใช่)
3. **กำหนดเส้นทางเวิร์กโฟลว์** โดยส่งข้อความไปยังตัวแทนที่เหมาะสม

คุณสมบัติสำคัญ:
- ใช้ตัวตกแต่ง `@handler` เพื่อเปิดเผยเมธอดเป็นขั้นตอนของเวิร์กโฟลว์
- รับ `RequestResponse[HumanFeedbackRequest, str]` ซึ่งมีทั้งคำขอต้นฉบับและคำตอบของผู้ใช้
- ส่งข้อความง่าย ๆ เป็น "ใช่" หรือ "ไม่ใช่" ซึ่งกระตุ้นฟังก์ชันเงื่อนไขของเรา


In [ ]:
class DecisionManager(Executor):
    """
    Coordinates workflow routing based on human feedback.
    
    This executor receives RequestResponse objects from the RequestInfoExecutor
    and makes routing decisions by sending simple messages that trigger
    condition functions.
    """

    def __init__(self, id: str | None = None):
        super().__init__(id=id or "decision_manager")

    @handler
    async def on_confirmation(
        self,
        response: AgentExecutorResponse,
        ctx: WorkflowContext,
    ) -> None:
        """Parse the confirmation question and pause the workflow for human input."""
        confirmation = ConfirmationQuestion.model_validate_json(response.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
                <strong>🔄 Requesting human input:</strong> {confirmation.question}
            </div>
        """)
        )
        # Pause the workflow; the human reply (a string) is delivered to on_human_feedback below.
        await ctx.request_info(
            request_data=HumanFeedbackRequest(
                prompt=confirmation.question,
                destination=confirmation.destination,
            ),
            response_type=str,
        )

    @response_handler
    async def on_human_feedback(
        self,
        original_request: HumanFeedbackRequest,
        feedback: str,
        ctx: WorkflowContext[AgentExecutorRequest, str],
    ) -> None:
        """Route the workflow based on the human's yes/no reply."""
        user_reply = (feedback or "").strip().lower()
        destination = original_request.destination or "unknown"

        display(
            HTML(f"""
            <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
                <strong>🎯 Decision Manager:</strong> Processing user reply: <strong>"{user_reply}"</strong> for {destination}
            </div>
        """)
        )

        if user_reply == "yes":
            display(
                HTML("""
                <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                    <strong>➡️  Routing:</strong> User wants alternatives → Will route to alternative_agent
                </div>
            """)
            )
            # Create and send a message for the alternative_agent
            user_msg = Message(
                role="user",
                contents=[f"The user wants to see alternative destinations near {destination}. Please suggest one."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        
        elif user_reply == "no":
            display(
                HTML("""
                <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 Routing:</strong> User declined alternatives → cancellation_agent
                </div>
            """)
            )
            user_msg = Message(
                role="user",
                contents=["The user has declined to see alternatives. Please acknowledge their decision."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        else:
            # Handle unexpected input - treat as decline
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                    <strong>⚠️  Warning:</strong> Unexpected input "{user_reply}" - treating as decline
                </div>
            """)
            )
            user_msg = Message(
                role="user",
                contents=["The user has declined to see alternatives. Please acknowledge their decision."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))


print("✅ DecisionManager executor created (@handler pauses via request_info, @response_handler routes)")

✅ DecisionManager executor created with @handler method for human feedback


## ขั้นตอนที่ 6: สร้างตัวประมวลผลการแสดงผลแบบกำหนดเอง

ตัวประมวลผลการแสดงผลเดียวกันจากเวิร์กโฟลว์เชิงเงื่อนไข - ให้ผลลัพธ์สุดท้ายเป็นผลลัพธ์ของเวิร์กโฟลว์


In [26]:
@executor(id="prepare_human_request")
async def prepare_human_request(
    response: AgentExecutorResponse, 
    ctx: WorkflowContext[HumanFeedbackRequest]
) -> None:
    """
    Transform agent response into HumanFeedbackRequest for RequestInfoExecutor.
    
    This executor bridges the type gap between:
    - confirmation_agent outputs AgentExecutorResponse with ConfirmationQuestion JSON
    - request_info_executor expects HumanFeedbackRequest (RequestInfoMessage dataclass)
    """
    display(
        HTML("""
        <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Transform:</strong> Converting ConfirmationQuestion to HumanFeedbackRequest
        </div>
    """)
    )
    
    # Parse the agent's Pydantic output (ConfirmationQuestion)
    confirmation = ConfirmationQuestion.model_validate_json(response.agent_run_response.text)
    
    # Convert to HumanFeedbackRequest dataclass for RequestInfoExecutor
    feedback_request = HumanFeedbackRequest(
        prompt=confirmation.question,
        destination=confirmation.destination
    )
    
    # Send the properly typed RequestInfoMessage to the RequestInfoExecutor
    await ctx.send_message(feedback_request)


@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ prepare_human_request executor created with @executor decorator")
print("✅ display_result executor created with @executor decorator")

✅ prepare_human_request executor created with @executor decorator
✅ display_result executor created with @executor decorator


## ขั้นตอนที่ 7: โหลดตัวแปรสภาพแวดล้อม

กำหนดค่าลูกค้า LLM (Microsoft Foundry / Azure OpenAI)


In [ ]:
# Load environment variables
load_dotenv()

# Microsoft Foundry provider with keyless AzureCliCredential auth (run `az login`).
# Matches the pattern used across lessons 01-13 and the other Lesson 14 notebooks.
chat_client = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)

print("✅ Chat client configured with Microsoft Foundry")


✅ Chat client configured with GitHub Models


## ขั้นตอนที่ 8: สร้างเอเจนต์และผู้ประมวลผล AI

เราสร้าง **ส่วนประกอบเวิร์กโฟลว์หกส่วน**:

**เอเจนต์ (ห่อหุ้มใน AgentExecutor):**
1. **availability_agent** - ตรวจสอบความพร้อมของโรงแรมโดยใช้เครื่องมือ
2. **confirmation_agent** - 🆕 เตรียมคำขอยืนยันจากมนุษย์
3. **alternative_agent** - แนะนำเมืองทางเลือก (เมื่อผู้ใช้ตอบตกลง)
4. **booking_agent** - กระตุ้นให้จอง (เมื่อมีห้องว่าง)
5. **cancellation_agent** - 🆕 จัดการข้อความยกเลิก (เมื่อผู้ใช้ตอบปฏิเสธ)

**ผู้ประมวลผลพิเศษ:**
6. **request_info_executor** - 🆕 `RequestInfoExecutor` ที่หยุดเวิร์กโฟลว์เพื่อรอข้อมูลจากมนุษย์
7. **decision_manager** - 🆕 ผู้ประมวลผลที่กำหนดเองที่กำหนดเส้นทางตามการตอบกลับของมนุษย์ (กำหนดไว้ก่อนหน้านี้แล้ว)


In [ ]:
# Agent 1: Check availability with tool (same as conditional workflow)
availability_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
    ),
    id="availability_agent",
)

# Agent 2: NEW - Prepare human confirmation request
confirmation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user's requested destination has no available hotel rooms. "
            "Create a polite message asking if they would like to see alternative destinations nearby. "
            "Return a JSON with: destination (the unavailable city), and question (a friendly yes/no question). "
            "Keep the question concise and friendly."
        ),
        default_options={"response_format": ConfirmationQuestion},  # Structured JSON output
    ),
    id="confirmation_agent",
)

# Agent 3: Suggest alternative (when user says yes)
alternative_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 4: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

# Agent 5: NEW - Handle cancellation when user declines alternatives
class CancellationMessage(BaseModel):
    """Message when user declines alternatives."""
    status: str
    message: str

cancellation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user has declined to see alternative hotel destinations. "
            "Create a polite cancellation message. "
            "Return JSON with: status (should be 'cancelled'), and message (a friendly acknowledgment). "
            "Keep the message brief and understanding."
        ),
        default_options={"response_format": CancellationMessage},
    ),
    id="cancellation_agent",
)

# DecisionManager instance - pauses for human input (request_info) and routes on the reply
decision_manager = DecisionManager(id="decision_manager")

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created Workflow Components:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>confirmation_agent</strong> 🆕 - Prepares human confirmation request</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
            <li><strong>cancellation_agent</strong> 🆕 - Handles user declining alternatives</li>
            <li><strong>request_info_executor</strong> 🆕 - Pauses workflow for human input</li>
            <li><strong>decision_manager</strong> 🆕 - Routes based on human response</li>
        </ul>
    </div>
""")
)


## ขั้นตอนที่ 9: สร้างเวิร์กโฟลว์ด้วย Human-in-the-Loop

ตอนนี้เราจะสร้างกราฟเวิร์กโฟลว์ด้วย **การจัดเส้นทางแบบมีเงื่อนไข** รวมถึงเส้นทาง human-in-the-loop:

**โครงสร้างเวิร์กโฟลว์:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙                    ↘
[no_availability]        [has_availability]
        ↓                        ↓
confirmation_agent          booking_agent
        ↓                        ↓
prepare_human_request      display_result
        ↓
request_info_executor (PAUSE)
        ↓
decision_manager
   ↙         ↘
[yes]        [no]
   ↓           ↓
alternative  cancellation
   ↓           ↓
display_result
```

**ขอบเขตสำคัญ:**
- `availability_agent → confirmation_agent` (เมื่อไม่มีห้อง)
- `confirmation_agent → prepare_human_request` (แปลงประเภท)
- `prepare_human_request → request_info_executor` (หยุดรอมนุษย์)
- `request_info_executor → decision_manager` (เสมอ - ให้ RequestResponse)
- `decision_manager → alternative_agent` (เมื่อผู้ใช้ตอบว่า "ใช่")
- `decision_manager → cancellation_agent` (เมื่อผู้ใช้ตอบว่า "ไม่")
- `availability_agent → booking_agent` (เมื่อมีห้องว่าง)
- เส้นทางทั้งหมดสิ้นสุดที่ `display_result`


In [ ]:
# Build the workflow with human-in-the-loop routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    
    # NO AVAILABILITY PATH (with human-in-the-loop)
    .add_edge(availability_agent, confirmation_agent, condition=no_availability_condition)
    .add_edge(confirmation_agent, decision_manager)  # decision_manager pauses via ctx.request_info
    
    # Decision manager routes based on user response
    .add_edge(decision_manager, alternative_agent, condition=user_wants_alternatives_condition)
    .add_edge(decision_manager, cancellation_agent, condition=user_declines_alternatives_condition)
    .add_edge(alternative_agent, display_result)
    .add_edge(cancellation_agent, display_result)
    
    # HAS AVAILABILITY PATH (no human input needed)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Human-in-the-Loop Routing:</strong><br>
            • If <strong>NO availability</strong> → confirmation_agent → prepare_human_request → request_info_executor → <strong>PAUSE FOR HUMAN</strong> → decision_manager<br>
            &nbsp;&nbsp;• If user says <strong>YES</strong> → alternative_agent → display_result<br>
            &nbsp;&nbsp;• If user says <strong>NO</strong> → cancellation_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result (no human input needed)
        </p>
    </div>
""")
)

## ขั้นตอนที่ 10: รันกรณีทดสอบ 1 - เมืองที่ไม่มีความพร้อม (ปารีสพร้อมการยืนยันจากมนุษย์)

การทดสอบนี้แสดงถึง **วงจรเต็มของมนุษย์ในลูป**:

1. ขอห้องพักในปารีส
2. availability_agent ตรวจสอบ → ไม่มีห้องว่าง
3. confirmation_agent สร้างคำถามสำหรับมนุษย์
4. request_info_executor **หยุดชั่วคราวการทำงาน** และส่ง `RequestInfoEvent`
5. **แอปพลิเคชันตรวจจับเหตุการณ์และแจ้งผู้ใช้ในคอนโซล**
6. ผู้ใช้พิมพ์ "ใช่" หรือ "ไม่ใช่"
7. แอปพลิเคชันส่งคำตอบผ่าน `send_responses_streaming()`
8. decision_manager นำทางตามคำตอบ
9. แสดงผลลัพธ์สุดท้าย

**รูปแบบสำคัญ:**
- ใช้ `workflow.run_stream()` สำหรับการวนรอบครั้งแรก
- ใช้ `workflow.send_responses_streaming(pending_responses)` สำหรับการวนรอบถัดไป
- ฟัง `RequestInfoEvent` เพื่อตรวจจับเมื่อจำเป็นต้องมีการป้อนข้อมูลจากมนุษย์
- ฟัง `WorkflowOutputEvent` เพื่อจับผลลัพธ์สุดท้าย


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability - Human-in-the-Loop)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → confirmation_agent → request_info_executor → <strong>PAUSE</strong> → decision_manager → (depends on user input)</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Paris"])], 
    should_respond=True
)

# Human-in-the-loop execution pattern.
# We script the human's answer ("yes") instead of input() so the notebook runs unattended.
# In a real app, replace SCRIPTED_ANSWER with input() or a UI callback.
SCRIPTED_ANSWER = "yes"
workflow_output: str | None = None

print("\n🔄 Starting human-in-the-loop workflow...")
print("=" * 60)

# First run streams events; resume with run(stream=True, responses=...) after each pause.
stream = workflow.run(request_paris, stream=True)
while True:
    requests: list[tuple[str, HumanFeedbackRequest]] = []
    async for event in stream:
        if event.type == "request_info" and isinstance(event.data, HumanFeedbackRequest):
            print(f"\n⏸️  WORKFLOW PAUSED - Human input requested!")
            print(f"   Request ID: {event.request_id}")
            print(f"   Destination: {event.data.destination}")
            print(f"   Question: {event.data.prompt}")
            requests.append((event.request_id, event.data))
        elif event.type == "output":
            workflow_output = str(event.data)
            print(f"\n✅ Workflow completed with output!")

    if not requests:
        break

    # Provide the (scripted) human answer for each pending request.
    responses: dict[str, str] = {}
    for req_id, req in requests:
        answer = SCRIPTED_ANSWER
        print(f"\n📝 (scripted) You answered: {answer}")
        responses[req_id] = answer

    stream = workflow.run(stream=True, responses=responses)

print(f"\n{'='*60}")
print(f"🏆 FINAL WORKFLOW OUTPUT:")
print(f"{'='*60}")

# Display final result
if workflow_output:
    # Try to parse as JSON for pretty display
    try:
        result_data = json.loads(workflow_output)
        if "alternative_destination" in result_data:
            result_obj = AlternativeResult.model_validate_json(workflow_output)
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> ✅ Accepted alternatives</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_obj.alternative_destination}</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_obj.reason}</p>
                    </div>
                </div>
            """)
            )
        else:
            # User declined
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #f44336 0%, #e91e63 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(244,67,54,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> 🚫 Declined alternatives</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Result:</strong> Booking request cancelled</p>
                    </div>
                </div>
            """)
            )
    except:
        print(workflow_output)


🔄 Starting human-in-the-loop workflow...

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: 032c8fce-b9d1-400e-ba8d-afd2248e2926
   Destination: Paris

💬 QUESTION FOR YOU:
   Unfortunately, there are no rooms available in Paris. Would you like to explore nearby alternative destinations?

📝 You answered: yes

📤 Sending human responses: {'032c8fce-b9d1-400e-ba8d-afd2248e2926': 'yes'}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'

📝 You answered: yes

📤 Sending human responses: {'032c8fce-b9d1-400e-ba8d-afd2248e2926': 'yes'}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: cf48dad0-ee5e-4f60-8806-341a7a292bd4
   Destination: Paris

💬 QUESTION FOR YOU:
   I'm sorry to inform you that there are no available hotel rooms in Paris. Would you like me to suggest nearby alternative destinations?

📝 You answered: 

📤 Sending human responses: {'cf48dad0-ee5e-4f60-8806-341a7a292bd4': ''}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'

📝 You answered: 

📤 Sending human responses: {'cf48dad0-ee5e-4f60-8806-341a7a292bd4': ''}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'


## ขั้นตอนที่ 11: รันกรณีทดสอบที่ 2 - เมืองที่มีห้องว่าง (สตอกโฮล์ม - ไม่ต้องการการป้อนข้อมูลจากมนุษย์)

การทดสอบนี้แสดงให้เห็นถึง **เส้นทางตรง** เมื่อห้องพักมีว่าง:

1. ขอห้องพักในสตอกโฮล์ม
2. ตัวแทนตรวจสอบห้องว่าง → มีห้องว่าง ✅
3. ตัวแทนจองแนะนำการจอง
4. แสดงผลลัพธ์ยืนยัน
5. **ไม่ต้องการการป้อนข้อมูลจากมนุษย์!**

กระบวนการนี้ข้ามเส้นทางที่มีมนุษย์ร่วมเข้าไปมีส่วนอย่างสิ้นเชิงเมื่อห้องพักมีว่าง


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability - No Human Input)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result (direct, no pause)</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Stockholm"])], 
    should_respond=True
)

# Run the workflow (should complete without human input)
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm - No Human Input)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0 0 10px 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <p style='margin: 10px 0 0 0; font-size: 12px; color: #999; font-style: italic;'>Note: No human input was requested because rooms were available!</p>
            </div>
        </div>
    """)
    )

## ประเด็นสำคัญและแนวทางปฏิบัติที่ดีที่สุดสำหรับ Human-in-the-Loop

### ✅ สิ่งที่คุณได้เรียนรู้:

#### 1. **รูปแบบ RequestInfoExecutor**
รูปแบบ human-in-the-loop ใน Microsoft Agent Framework ใช้สามส่วนประกอบหลัก:
- `RequestInfoExecutor` - หยุดชั่วคราวของเวิร์กโฟลว์และส่งอีเวนต์
- `RequestInfoMessage` - คลาสฐานสำหรับข้อมูลร้องขอแบบพิมพ์ (สืบทอดคลาสนี้!)
- `RequestResponse` - เชื่อมโยงคำตอบของมนุษย์กับคำขอดั้งเดิม

**ความเข้าใจที่สำคัญ:**
- `RequestInfoExecutor` ไม่ได้เก็บข้อมูลอินพุตเอง - เพียงแค่หยุดเวิร์กโฟลว์เท่านั้น
- โค้ดแอปของคุณต้องฟัง `RequestInfoEvent` และเก็บข้อมูลอินพุต
- คุณต้องเรียกใช้ `send_responses_streaming()` พร้อมพจนานุกรมแมป `request_id` กับคำตอบของผู้ใช้

#### 2. **รูปแบบ Streaming Execution**
```python
# การวนรอบครั้งแรก
stream = workflow.run_stream(initial_request)

# การวนรอบถัดไป (หลังจากได้รับข้อมูลจากมนุษย์)
stream = workflow.send_responses_streaming(pending_responses)

# ประมวลผลเหตุการณ์เสมอ
events = [event async for event in stream]
```

#### 3. **สถาปัตยกรรมแบบขับเคลื่อนด้วยเหตุการณ์**
ฟังอีเวนต์เฉพาะเพื่อควบคุมเวิร์กโฟลว์:
- `RequestInfoEvent` - ต้องการข้อมูลอินพุตจากมนุษย์ (เวิร์กโฟลว์หยุดชั่วคราว)
- `WorkflowOutputEvent` - ผลลัพธ์สุดท้ายพร้อมใช้งาน (เวิร์กโฟลว์เสร็จสมบูรณ์)
- `WorkflowStatusEvent` - การเปลี่ยนแปลงสถานะ (IN_PROGRESS, IDLE_WITH_PENDING_REQUESTS ฯลฯ)

#### 4. **Executor ที่กำหนดเองกับ @handler**
`DecisionManager` แสดงตัวอย่างวิธีสร้าง executor ที่:
- ใช้ตัวตกแต่ง `@handler` เพื่อเปิดเผยเมธอดเป็นขั้นตอนของเวิร์กโฟลว์
- รับข้อความแบบพิมพ์ (เช่น `RequestResponse[HumanFeedbackRequest, str]`)
- นำทางเวิร์กโฟลว์โดยส่งข้อความไปยัง executor อื่นๆ
- เข้าถึง context ผ่าน `WorkflowContext`

#### 5. **การนำทางแบบมีเงื่อนไขด้วยการตัดสินใจของมนุษย์**
คุณสามารถสร้างฟังก์ชันเงื่อนไขที่ประเมินคำตอบของมนุษย์:
```python
def user_wants_alternatives_condition(message: Any) -> bool:
    response_text = message.agent_run_response.text.lower()
    return response_text == "yes"
```

### 🎯 การใช้งานในโลกจริง:

1. **เวิร์กโฟลว์การอนุมัติ**
   - ขอนายอนุมัติก่อนดำเนินการรายงานค่าใช้จ่าย
   - ต้องการการตรวจสอบจากมนุษย์ก่อนส่งอีเมลอัตโนมัติ
   - ยืนยันรายการธุรกรรมมูลค่าสูงก่อนดำเนินการ

2. **การควบคุมเนื้อหา**
   - ทำเครื่องหมายเนื้อหาที่น่าสงสัยเพื่อให้มนุษย์ตรวจสอบ
   - ขอให้ผู้ควบคุมเนื้อหาตัดสินใจในกรณีที่ยาก
   - เพิ่มขั้นตอนให้มนุษย์เมื่อความมั่นใจของ AI ต่ำ

3. **บริการลูกค้า**
   - ให้ AI จัดการคำถามทั่วไปโดยอัตโนมัติ
   - ส่งต่อปัญหาซับซ้อนให้ตัวแทนมนุษย์
   - ถามลูกค้าว่าต้องการพูดคุยกับมนุษย์หรือไม่

4. **การประมวลผลข้อมูล**
   - ขอให้มนุษย์แก้ไขข้อมูลที่คลุมเครือ
   - ยืนยันการตีความของ AI ต่อเอกสารที่ไม่ชัดเจน
   - ให้ผู้ใช้เลือกจากการตีความที่ถูกต้องหลายแบบ

5. **ระบบความปลอดภัยสำคัญ**
   - ต้องการการยืนยันจากมนุษย์ก่อนการกระทำที่ไม่สามารถย้อนกลับได้
   - ขออนุมัติก่อนเข้าถึงข้อมูลที่ละเอียดอ่อน
   - ยืนยันการตัดสินใจในอุตสาหกรรมที่มีการกำกับดูแล (สุขภาพ, การเงิน)

6. **เอเจนต์โต้ตอบ**
   - สร้างบ็อตสนทนาที่ถามคำถามเพิ่มเติม
   - สร้างวิซาร์ดที่แนะนำผู้ใช้ผ่านกระบวนการซับซ้อน
   - ออกแบบเอเจนต์ที่ทำงานร่วมกับมนุษย์ทีละขั้นตอน

### 🔄 การเปรียบเทียบ: มี vs ไม่มี Human-in-the-Loop

| คุณสมบัติ | เวิร์กโฟลว์แบบมีเงื่อนไข | เวิร์กโฟลว์ Human-in-the-Loop |
|---------|---------------------|---------------------------|
| **การดำเนินการ** | เรียกใช้ `workflow.run()` ครั้งเดียว | ลูปด้วย `run_stream()` + `send_responses_streaming()` |
| **ข้อมูลอินพุตผู้ใช้** | ไม่มี (อัตโนมัติเต็มรูปแบบ) | คำสั่งโต้ตอบผ่าน `input()` หรือ UI |
| **ส่วนประกอบ** | Agents + Executors | + RequestInfoExecutor + DecisionManager |
| **อีเวนต์** | AgentExecutorResponse เท่านั้น | RequestInfoEvent, WorkflowOutputEvent ฯลฯ |
| **การหยุดชั่วคราว** | ไม่มีการหยุดชั่วคราว | เวิร์กโฟลว์หยุดที่ RequestInfoExecutor |
| **การควบคุมโดยมนุษย์** | ไม่มีการควบคุมโดยมนุษย์ | มนุษย์ตัดสินใจสำคัญ |
| **กรณีการใช้งาน** | อัตโนมัติ | ความร่วมมือและการควบคุม |

### 🚀 รูปแบบขั้นสูง:

#### จุดตัดสินใจหลายจุดโดยมนุษย์
คุณสามารถมีโหนด `RequestInfoExecutor` หลายจุดในเวิร์กโฟลว์เดียวกันได้:
```python
.add_edge(agent1, request_info_1)  # การตัดสินใจของมนุษย์ครั้งแรก
.add_edge(decision_manager_1, agent2)
.add_edge(agent2, request_info_2)  # การตัดสินใจของมนุษย์ครั้งที่สอง
.add_edge(decision_manager_2, final_agent)
```

#### การจัดการเวลาหมดเวลา
ใช้เวลาหมดเวลาสำหรับคำตอบของมนุษย์:
```python
import asyncio

try:
    answer = await asyncio.wait_for(
        asyncio.to_thread(input, "Enter yes/no: "),
        timeout=60.0
    )
except asyncio.TimeoutError:
    answer = "no"  # ค่าเริ่มต้นเป็นตัวเลือกที่ปลอดภัย
```

#### การรวม UI อย่างสมบูรณ์
แทนที่จะใช้ `input()` ผนวกกับเว็บ UI, Slack, Teams ฯลฯ:
```python
if isinstance(event, RequestInfoEvent):
    # ส่งการแจ้งเตือนไปยังช่องทางที่ผู้ใช้ต้องการ
    await slack_client.send_message(
        user_id=current_user,
        text=event.data.prompt,
        request_id=event.request_id
    )
```

#### เงื่อนไข Human-in-the-Loop
ขอข้อมูลอินพุตจากมนุษย์ในสถานการณ์เฉพาะเท่านั้น:
```python
def needs_human_approval_condition(message: Any) -> bool:
    # ส่งต่อไปยังมนุษย์เฉพาะเมื่อความมั่นใจต่ำหรือค่ามีค่าสูง
    if result.confidence < 0.7 or result.value > 10000:
        return True
    return False
```

### ⚠️ แนวทางปฏิบัติที่ดีที่สุด:

1. **สืบทอด RequestInfoMessage เสมอ**
   - ให้ความปลอดภัยของชนิดข้อมูลและการตรวจสอบ
   - เปิดใช้งานบริบทที่สมบูรณ์สำหรับการแสดงผล UI
   - ช่วยชัดเจนเจตนาของแต่ละประเภทคำขอ

2. **ใช้คำสั่งที่อธิบายชัดเจน**
   - รวมบริบทเกี่ยวกับสิ่งที่คุณถาม
   - อธิบายผลของแต่ละตัวเลือก
   - รักษาคำถามให้ง่ายและชัดเจน

3. **จัดการกับอินพุตที่ไม่คาดคิด**
   - ตรวจสอบคำตอบของผู้ใช้
   - ให้ค่าดีฟอลต์สำหรับอินพุตที่ไม่ถูกต้อง
   - ให้ข้อความแสดงข้อผิดพลาดที่ชัดเจน

4. **ติดตามรหัสคำขอ**
   - ใช้ความสัมพันธ์ระหว่าง request_id กับคำตอบ
   - อย่าพยายามจัดการสถานะด้วยตนเอง

5. **ออกแบบให้ไม่บล็อก**
   - อย่าบล็อกเธรดขณะรอข้อมูลอินพุต
   - ใช้รูปแบบ async ตลอดเวลา
   - รองรับอินสแตนซ์เวิร์กโฟลว์ทำงานพร้อมกัน

### 📚 แนวคิดที่เกี่ยวข้อง:

- **Agent Middleware** - ดักจับการเรียก agent (สมุดบันทึกก่อนหน้า)
- **การจัดการสถานะเวิร์กโฟลว์** - เก็บสถานะเวิร์กโฟลว์ระหว่างรัน
- **ความร่วมมือหลายเอเจนต์** - รวม human-in-the-loop กับทีม agent
- **สถาปัตยกรรมขับเคลื่อนด้วยเหตุการณ์** - สร้างระบบตอบสนองด้วยอีเวนต์

---

### 🎓 ยินดีด้วย!

คุณได้เชี่ยวชาญเวิร์กโฟลว์ human-in-the-loop ด้วย Microsoft Agent Framework แล้ว! ตอนนี้คุณรู้วิธีที่จะ:
- ✅ หยุดเวิร์กโฟลว์เพื่อรวบรวมข้อมูลอินพุตจากมนุษย์
- ✅ ใช้ RequestInfoExecutor และ RequestInfoMessage
- ✅ จัดการการดำเนินการแบบสตรีมด้วยอีเวนต์
- ✅ สร้าง executor กำหนดเองด้วย @handler
- ✅ นำทางเวิร์กโฟลว์โดยอาศัยการตัดสินใจของมนุษย์
- ✅ สร้างเอเจนต์ AI โต้ตอบที่ทำงานร่วมกับมนุษย์

**นี่คือรูปแบบสำคัญสำหรับการสร้างระบบ AI ที่เชื่อถือได้และควบคุมได้!** 🚀


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ปฏิเสธความรับผิดชอบ**:
เอกสารนี้ได้รับการแปลโดยใช้บริการแปลภาษา AI [Co-op Translator](https://github.com/Azure/co-op-translator) ขณะที่เราพยายามให้ความถูกต้อง โปรดทราบว่าการแปลโดยอัตโนมัติอาจมีข้อผิดพลาดหรือความไม่ถูกต้อง เอกสารต้นฉบับในภาษาต้นทางควรถูกพิจารณาเป็นแหล่งข้อมูลที่เชื่อถือได้ สำหรับข้อมูลที่สำคัญ แนะนำให้ใช้การแปลโดยมนุษย์มืออาชีพ เราไม่รับผิดชอบต่อความเข้าใจผิดหรือการตีความที่ผิดพลาดที่เกิดขึ้นจากการใช้การแปลนี้
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
